# Bivariate-bicycle CSS decoding

Load a small parity-check fixture, enlarge its CSS generators in memory, decode one sector, and run a short seeded sweep. Generated enlarged matrices are not written to the repository.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

using NPZ
using SparseArrays

In [ ]:
fixture_path = joinpath(
    repo_root,
    "examples",
    "fixtures",
    "bivariate_bicycle",
    "30_4_5_p2.npz",
)
loaded = load_sparse_quantum_code(fixture_path; hx_key=:hx, hz_key=:hz)
enlarged = enlarge_css_generators(
    loaded.Hx,
    loaded.Hz;
    p=2,
    method=:systematic,
    progress_io=nothing,
)
Mqq = Float64.(enlarged.Mqq) ./ sqrt(2)
Mpp = Float64.(enlarged.Mpp) ./ sqrt(2)

H = Mqq
G = inv(Mpp)
logical_check = inv(Mqq)
problem = QuantumDecodingProblem(H, G, logical_check)

sigma = 0.10
rng = MersenneTwister(60)
error_vector = sample_error(rng, sigma, size(H, 2))
received = copy(error_vector)
decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=6,
)
soft_estimate = run_decoder!(decoder, received)
decision = hard_decision(soft_estimate, H)
residual = error_vector - (received - G * decision)

(
    source_check_sizes=(size(loaded.Hx), size(loaded.Hz)),
    enlarged_sizes=(size(Mqq), size(Mpp)),
    logical_error=is_logical_error(logical_check, residual),
)

In [ ]:
sigmas = [0.08, 0.10, 0.12]
estimates = [
    estimate_logical_error_rate!(
        MersenneTwister(600),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=6,
        ),
        problem;
        samples=samples_per_point,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        failures=result.events,
        samples=result.samples,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

For larger BB instances, run the same construction outside the repository and retain the code name, source, enlargement method, package commit, and random seed with the results.